# 05 février 2025

Ce notebook télécharge les OU du Sierra Leone et fait une analyse ultrabasique
Le but est de le transformer en pipeline

In [14]:
from openhexa.sdk import current_run

# Valeur par défaut (local / debug)
districts = [
    "Bonthe",
    "Kambia",
    "Koinadugu",
    "Port Loko",
    "Pujehun",
    "Tonkolili"
]

# Si exécuté dans une pipeline OpenHEXA
if current_run is not None:
    try:
        districts = current_run.get_parameter("districts")
    except Exception:
        pass

# Normalisation → toujours une liste
if isinstance(districts, str):
    district_list = [d.strip() for d in districts.split(",") if d.strip()]
elif isinstance(districts, (list, tuple)):
    district_list = list(districts)
else:
    raise ValueError("Format du paramètre 'districts' invalide")

district_list

['Bonthe', 'Kambia', 'Koinadugu', 'Port Loko', 'Pujehun', 'Tonkolili']

In [15]:
from openhexa.toolbox.dhis2 import DHIS2
from openhexa.toolbox.dhis2.dataframe import get_organisation_units

# ⚡ Objet DHIS2 directement
dhis = DHIS2(
    url="https://play.im.dhis2.org/stable-2-42-3-1/",
    username="admin",
    password="district"
)

df = get_organisation_units(dhis).to_pandas()

In [16]:
from pathlib import Path

output_dir = Path("../03_output")
output_dir.mkdir(exist_ok=True)

outputs = []

for district in district_list:
    print(f"📍 Traitement du district : {district}")

    df_district = df[df["level_2_name"] == district]

    if df_district.empty:
        print(f"⚠️ Aucun résultat pour {district}")
        continue

    df_summary = (
        df_district
        .groupby("level")
        .agg(
            n_ous=("id", "count"),
            n_with_gps=("geometry", lambda x: x.notnull().sum())
        )
        .reset_index()
    )

    df_summary["pct_with_gps"] = (
        df_summary["n_with_gps"] / df_summary["n_ous"] * 100
    ).round(1)

    output_file = output_dir / f"{district}_ou_gps_summary.csv"
    df_summary.to_csv(output_file, index=False)

    outputs.append(str(output_file))

📍 Traitement du district : Bonthe
📍 Traitement du district : Kambia
📍 Traitement du district : Koinadugu
📍 Traitement du district : Port Loko
📍 Traitement du district : Pujehun
📍 Traitement du district : Tonkolili


In [17]:
from openhexa.sdk import current_run

if current_run is not None:
    for f in outputs:
        current_run.add_file_output(f)

Sending output with path ../03_output/Bonthe_ou_gps_summary.csv
Sending output with path ../03_output/Kambia_ou_gps_summary.csv
Sending output with path ../03_output/Koinadugu_ou_gps_summary.csv
Sending output with path ../03_output/Port Loko_ou_gps_summary.csv
Sending output with path ../03_output/Pujehun_ou_gps_summary.csv
Sending output with path ../03_output/Tonkolili_ou_gps_summary.csv
